
# Tech Challenge — Fase 4
## Notebook 03 — Embeddings e Busca Vetorial

### Objetivo

Construir a etapa de recuperação semântica do sistema RAG, utilizando as avaliações textuais da Olist preparadas no Notebook 02.

### Base de conhecimento

- **Origem:** `data/processed/olist_reviews_documentos_rag.jsonl`
- **Quantidade:** 42.687 documentos textuais
- **Conteúdo:** avaliações de clientes, identificadores e metadados
- **Preparação:** realizada e validada no Notebook 02

### Etapas previstas

1. Carregar e validar os documentos preparados.
2. Selecionar um modelo de embeddings adequado aos textos em português.
3. Gerar representações vetoriais dos documentos.
4. Armazenar os vetores em uma base vetorial persistente.
5. Implementar e testar a recuperação semântica.
6. Preservar a identificação dos documentos e seus metadados para rastreabilidade.

### Decisões metodológicas

- Reutilizar a base preparada, sem repetir a limpeza dos textos.
- Preservar os documentos originais e seus identificadores.
- Processar os embeddings em lotes, considerando o volume de dados.
- Verificar a qualidade das buscas antes de integrar o modelo de linguagem.
- Manter a implementação reproduzível e alinhada aos requisitos da Fase 4.

### Resultado esperado

Uma base vetorial consultável, capaz de recuperar avaliações semanticamente relevantes para fundamentar as respostas do futuro sistema RAG.

**Observação:** similaridade semântica não garante relevância ou correção factual. A recuperação deverá ser avaliada por meio de perguntas e evidências verificáveis.

### Nota da adaptação ao Databricks
A sequência e as explicações deste notebook foram preservadas a partir da versão organizada original.
O JSONL preparado pelo Notebook 02 é lido em
`/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase4/data/processed/`.
As bibliotecas são instaladas juntas na etapa já existente de instalação; uma célula técnica adicional
reinicia o Python, e a etapa seguinte recarrega os documentos.
Os resultados narrados no texto original correspondem à execução local anterior; confirme-os nas saídas
calculadas nesta execução Databricks. As saídas antigas foram removidas para não parecerem novas execuções.


In [0]:

# Objetivo:
# Importar as bibliotecas necessárias para localizar os arquivos do projeto.

# Justificativa:
# O notebook está na pasta "notebooks", enquanto a base preparada
# está em "data/processed". Precisamos localizar a raiz do projeto
# independentemente da pasta em que o Jupyter foi iniciado.

# Ação:
from pathlib import Path
import pandas as pd

# Localizar os dados no mesmo Volume utilizado pelos notebooks 00, 01 e 02.
DIRETORIO_ATUAL = Path.cwd().resolve()
BASE_DIR = Path(
    "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase4"
)
if not (BASE_DIR / "data" / "processed").is_dir():
    raise FileNotFoundError(
        f"Diretório não encontrado: {BASE_DIR / 'data' / 'processed'}. "
        "Confirme o Volume e a execução dos notebooks 00 a 02."
    )

# Definir o caminho da base de conhecimento preparada no Notebook 02.
ARQUIVO_DOCUMENTOS = (
    BASE_DIR / "data" / "processed" / "olist_reviews_documentos_rag.jsonl"
)

print(f"Diretório do projeto: {BASE_DIR}")
print(f"Arquivo de documentos: {ARQUIVO_DOCUMENTOS}")
print(f"Arquivo encontrado: {ARQUIVO_DOCUMENTOS.is_file()}")


In [0]:

# Objetivo:
# Carregar os documentos preparados no Notebook 02.

# Justificativa:
# Antes de gerar embeddings, precisamos verificar se a base
# de conhecimento foi carregada corretamente e se mantém
# a estrutura definida na etapa de preparação.

# Ação:
df_documentos_rag = pd.read_json(
    ARQUIVO_DOCUMENTOS,
    orient="records",
    lines=True,
    dtype={
        "documento_id": "str",
        "review_id": "str",
        "order_id": "str",
        "review_score": "int64",
        "review_creation_date": "str",
        "texto_documento": "str"
    },
    convert_dates=False
)

print(f"Quantidade de documentos: {len(df_documentos_rag):,}")
print(f"Quantidade de colunas: {df_documentos_rag.shape[1]}")

display(df_documentos_rag.head(3))


In [0]:

# Objetivo:
# Verificar a integridade da base de documentos antes da vetorização.

# Justificativa:
# Cada documento precisa ter um identificador único, texto válido
# e metadados essenciais para permitir recuperação e rastreabilidade.

# Ação:
colunas_obrigatorias = [
    "documento_id",
    "review_id",
    "order_id",
    "review_score",
    "review_creation_date",
    "texto_documento"
]

colunas_ausentes = [
    coluna
    for coluna in colunas_obrigatorias
    if coluna not in df_documentos_rag.columns
]

ids_duplicados = df_documentos_rag["documento_id"].duplicated().sum()

textos_vazios = (
    df_documentos_rag["texto_documento"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

valores_nulos = df_documentos_rag[colunas_obrigatorias].isna().sum()

print(f"Colunas obrigatórias ausentes: {colunas_ausentes}")
print(f"Identificadores duplicados: {ids_duplicados}")
print(f"Documentos com texto vazio: {textos_vazios}")
print("\nValores nulos por coluna:")
print(valores_nulos)


In [0]:

# Objetivo:
# Verificar se a biblioteca Sentence Transformers está instalada.

# Justificativa:
# A biblioteca será utilizada para transformar os textos das
# avaliações em embeddings, permitindo a busca semântica.

# Ação:
from importlib.util import find_spec

biblioteca_instalada = find_spec("sentence_transformers") is not None

print(f"Sentence Transformers instalada: {biblioteca_instalada}")

if biblioteca_instalada:
    print("A biblioteca está disponível no ambiente atual.")
else:
    print("A biblioteca ainda precisa ser instalada.")

In [0]:

# Objetivo:
# Instalar a biblioteca Sentence Transformers.

# Justificativa:
# A biblioteca permitirá carregar um modelo pré-treinado
# e gerar embeddings dos documentos da Olist.
#
# O comando %pip instala o pacote no ambiente Python
# utilizado pelo kernel atual do Jupyter.

# Ação:
%pip install "sentence-transformers==5.1.0" "chromadb==1.0.20"

In [0]:
# Databricks: reiniciar o Python para disponibilizar as bibliotecas instaladas.
# A célula seguinte recupera os caminhos e os documentos antes de continuar.
dbutils.library.restartPython()


In [0]:

# Objetivo:
# Confirmar a instalação da biblioteca Sentence Transformers.

# Justificativa:
# Antes de selecionar e carregar um modelo de embeddings,
# precisamos verificar se a biblioteca pode ser importada
# corretamente pelo kernel do notebook.

# Ação:
# Databricks: recuperar os caminhos e os documentos após reiniciar o Python.
from pathlib import Path
import pandas as pd
BASE_DIR = Path(
    "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase4"
)
ARQUIVO_DOCUMENTOS = BASE_DIR / "data" / "processed" / "olist_reviews_documentos_rag.jsonl"
df_documentos_rag = pd.read_json(
    ARQUIVO_DOCUMENTOS, orient="records", lines=True, convert_dates=False,
    dtype={"documento_id": "str", "review_id": "str", "order_id": "str",
           "review_score": "int64", "review_creation_date": "str", "texto_documento": "str"}
)

from importlib.metadata import version

from sentence_transformers import SentenceTransformer

versao_sentence_transformers = version("sentence-transformers")

print(f"Sentence Transformers: {versao_sentence_transformers}")
print("Importação realizada com sucesso!")


In [0]:

# Objetivo:
# Carregar um modelo multilíngue de embeddings.

# Justificativa:
# As avaliações da Olist estão predominantemente em português.
# Precisamos de um modelo capaz de representar semanticamente
# esses textos para permitir buscas por similaridade.

# Ação:
NOME_MODELO = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

modelo_embeddings = SentenceTransformer(NOME_MODELO)

print(f"Modelo carregado: {NOME_MODELO}")
print(f"Dimensões dos embeddings: {modelo_embeddings.get_sentence_embedding_dimension()}")

In [0]:

# Objetivo:
# Gerar e inspecionar nosso primeiro embedding.

# Justificativa:
# Antes de processar toda a base da Olist, precisamos
# compreender a estrutura numérica produzida pelo modelo.

# Ação:
texto_exemplo = "Meu pedido chegou muito atrasado."

embedding_exemplo = modelo_embeddings.encode(texto_exemplo)

print(f"Texto original: {texto_exemplo}")
print(f"Tipo do resultado: {type(embedding_exemplo)}")
print(f"Formato do embedding: {embedding_exemplo.shape}")
print(f"Quantidade de dimensões: {len(embedding_exemplo)}")

print("\nPrimeiros 10 valores do embedding:")
print(embedding_exemplo[:10])

In [0]:

# Objetivo:
# Comparar a similaridade semântica entre avaliações de clientes.

# Justificativa:
# A busca vetorial utiliza os embeddings para identificar textos
# potencialmente relacionados, mesmo quando empregam palavras diferentes.

# Ação:
from sentence_transformers import util

avaliacoes_exemplo = [
    "Meu pedido chegou muito atrasado.",
    "A encomenda demorou demais para ser entregue.",
    "O produto tem excelente qualidade."
]

# Gerar um embedding para cada avaliação.
embeddings_avaliacoes = modelo_embeddings.encode(avaliacoes_exemplo)

# Comparar a primeira avaliação com as demais.
similaridade_atraso = util.cos_sim(
    embeddings_avaliacoes[0],
    embeddings_avaliacoes[1]
).item()

similaridade_qualidade = util.cos_sim(
    embeddings_avaliacoes[0],
    embeddings_avaliacoes[2]
).item()

print("Avaliação de referência:")
print(avaliacoes_exemplo[0])

print("\nComparação com a avaliação sobre demora:")
print(f"Similaridade de cosseno: {similaridade_atraso:.4f}")

print("\nComparação com a avaliação sobre qualidade:")
print(f"Similaridade de cosseno: {similaridade_qualidade:.4f}")


In [0]:

# Objetivo:
# Simular uma busca semântica com uma pergunta e três avaliações.

# Justificativa:
# Em um sistema RAG, a pergunta do usuário também é
# transformada em embedding para ser comparada com
# os embeddings dos documentos disponíveis.

# Ação:
pergunta_exemplo = "Quais clientes reclamaram de atraso na entrega?"

documentos_exemplo = [
    "O pedido chegou depois do prazo prometido.",
    "Gostei muito da qualidade e do acabamento do produto.",
    "Recebi o produto antes da data prevista."
]

# Gerar o embedding da pergunta.
embedding_pergunta = modelo_embeddings.encode(pergunta_exemplo)

# Gerar os embeddings dos documentos fictícios.
embeddings_documentos = modelo_embeddings.encode(documentos_exemplo)

# Calcular a similaridade entre a pergunta e cada documento.
similaridades = util.cos_sim(
    embedding_pergunta,
    embeddings_documentos
)[0]

# Organizar os resultados da maior para a menor similaridade.
resultados = sorted(
    zip(documentos_exemplo, similaridades.tolist()),
    key=lambda item: item[1],
    reverse=True
)

print(f"Pergunta: {pergunta_exemplo}\n")

for posicao, (documento, similaridade) in enumerate(resultados, start=1):
    print(f"{posicao}º lugar | Similaridade: {similaridade:.4f}")
    print(f"Documento: {documento}\n")


In [0]:

# Objetivo:
# Gerar embeddings para uma pequena amostra de avaliações reais da Olist.

# Justificativa:
# Antes de processar os 42.687 documentos, precisamos validar
# se o modelo consegue transformar nossos textos preparados
# em vetores com as dimensões esperadas.

# Ação:

# Selecionar os cinco primeiros documentos da base.
amostra_documentos = df_documentos_rag.head(5).copy()

# Extrair os textos que serão enviados ao modelo.
textos_amostra = amostra_documentos["texto_documento"].tolist()

# Gerar os embeddings dos cinco documentos.
embeddings_amostra = modelo_embeddings.encode(textos_amostra)

# Exibir informações para validação.
print(f"Quantidade de documentos: {len(textos_amostra)}")
print(f"Formato da matriz de embeddings: {embeddings_amostra.shape}")

print("\nPrimeiro documento:")
print(textos_amostra[0])

print("\nPrimeiros 5 valores do seu embedding:")
print(embeddings_amostra[0][:5])


In [0]:

# Objetivo:
# Planejar o processamento dos documentos em lotes.

# Justificativa:
# O processamento em lotes permite controlar quantos textos
# são enviados simultaneamente ao modelo de embeddings.
# Antes de executar essa operação, vamos calcular quantos
# lotes serão necessários para nossa base de conhecimento.

# Ação:
from math import ceil

# Definir a quantidade de documentos por lote.
TAMANHO_LOTE = 64

# Consultar a quantidade total de documentos.
total_documentos = len(df_documentos_rag)

# Calcular a quantidade de lotes necessária.
total_lotes = ceil(total_documentos / TAMANHO_LOTE)

# Calcular a quantidade de documentos no último lote.
documentos_ultimo_lote = total_documentos % TAMANHO_LOTE

if documentos_ultimo_lote == 0:
    documentos_ultimo_lote = TAMANHO_LOTE

print(f"Total de documentos: {total_documentos:,}")
print(f"Documentos por lote: {TAMANHO_LOTE}")
print(f"Quantidade de lotes: {total_lotes}")
print(f"Documentos no último lote: {documentos_ultimo_lote}")


In [0]:

# Objetivo:
# Gerar embeddings para os 42.687 documentos da Olist.

# Justificativa:
# Cada documento precisa de uma representação vetorial
# para que possamos realizar buscas por similaridade semântica.
# O processamento será realizado em lotes para controlar
# a quantidade de textos processados simultaneamente.

# Ação:

# Extrair os textos da base de conhecimento.
textos_documentos = df_documentos_rag["texto_documento"].tolist()

# Gerar os embeddings de todos os documentos.
embeddings_documentos_rag = modelo_embeddings.encode(
    textos_documentos,
    batch_size=TAMANHO_LOTE,
    show_progress_bar=True,
    convert_to_numpy=True
)

# Validar as dimensões da matriz gerada.
print(f"Documentos processados: {len(textos_documentos):,}")
print(f"Formato da matriz: {embeddings_documentos_rag.shape}")
print(f"Tipo dos valores: {embeddings_documentos_rag.dtype}")

In [0]:

# Objetivo:
# Salvar os embeddings dos documentos da Olist em disco.

# Justificativa:
# Os embeddings foram gerados com custo computacional.
# Salvá-los permitirá reutilizá-los sem repetir o processamento,
# mesmo após reiniciar o kernel do notebook.

# Ação:
import numpy as np

# Definir o diretório e o arquivo de saída.
DIRETORIO_EMBEDDINGS = BASE_DIR / "data" / "processed"
ARQUIVO_EMBEDDINGS = DIRETORIO_EMBEDDINGS / "olist_reviews_embeddings.npy"

# Garantir que o diretório exista.
DIRETORIO_EMBEDDINGS.mkdir(parents=True, exist_ok=True)

# Salvar a matriz numérica no formato nativo do NumPy.
np.save(ARQUIVO_EMBEDDINGS, embeddings_documentos_rag)

# Conferir se o arquivo foi criado.
print(f"Arquivo: {ARQUIVO_EMBEDDINGS}")
print(f"Arquivo encontrado: {ARQUIVO_EMBEDDINGS.exists()}")
print(f"Tamanho: {ARQUIVO_EMBEDDINGS.stat().st_size / (1024 ** 2):.2f} MiB")

In [0]:

# Objetivo:
# Validar o arquivo de embeddings salvo em disco.

# Justificativa:
# Precisamos confirmar que a matriz pode ser recuperada
# sem alterações antes de utilizá-la no banco vetorial.

# Ação:

# Carregar os embeddings diretamente do arquivo .npy.
embeddings_carregados = np.load(
    ARQUIVO_EMBEDDINGS,
    allow_pickle=False
)

# Conferir as dimensões e o tipo dos dados.
print(f"Formato da matriz carregada: {embeddings_carregados.shape}")
print(f"Tipo dos valores: {embeddings_carregados.dtype}")

# Verificar se existe um embedding para cada documento.
quantidades_correspondem = (
    len(embeddings_carregados) == len(df_documentos_rag)
)

# Comparar a matriz carregada com a matriz original.
valores_identicos = np.array_equal(
    embeddings_carregados,
    embeddings_documentos_rag
)

print(f"Quantidade corresponde aos documentos: {quantidades_correspondem}")
print(f"Valores idênticos à matriz original: {valores_identicos}")

In [0]:

# Objetivo:
# Instalar a biblioteca ChromaDB.

# Justificativa:
# O ChromaDB permitirá armazenar os embeddings,
# associá-los aos documentos e seus metadados
# e realizar buscas por similaridade semântica.
#
# O comando %pip utiliza o ambiente Python
# associado ao kernel atual do Jupyter.

# Ação:
# No Databricks, o ChromaDB já foi instalado junto ao Sentence Transformers.
# Não reinstalar aqui: isso poderia exigir novo reinício do Python.
from importlib.metadata import version
print(f"ChromaDB já instalado na etapa de dependências: {version('chromadb')}")


## Etapa 3 — Banco Vetorial com ChromaDB

### Objetivo

Construir um banco vetorial local para armazenar os embeddings das avaliações da Olist e permitir a recuperação de documentos por similaridade semântica.

### Contexto

Na etapa anterior, utilizamos o modelo `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` para gerar embeddings dos **42.687 documentos** da nossa base de conhecimento.

Cada documento foi representado por um vetor de **384 dimensões**, e a matriz resultante foi salva e validada no arquivo:

`data/processed/olist_reviews_embeddings.npy`

Os textos originais, identificadores e metadados permanecem disponíveis em:

`data/processed/olist_reviews_documentos_rag.jsonl`

### Estratégia

Utilizaremos o **ChromaDB** para criar uma coleção vetorial persistente, associando cada embedding ao documento correspondente e aos seus metadados.

Nesta etapa, realizaremos:

1. Verificação da instalação do ChromaDB.
2. Carregamento dos documentos e embeddings previamente salvos.
3. Criação de uma coleção vetorial persistente.
4. Inserção dos documentos, identificadores, metadados e embeddings.
5. Validação da quantidade de registros armazenados.
6. Testes de recuperação por similaridade semântica.

### Resultado esperado

Ao final desta etapa, teremos uma base vetorial local capaz de recuperar avaliações relevantes para uma pergunta, fornecendo os documentos e seus metadados para as etapas seguintes do sistema RAG.

**Observação:** a recuperação por similaridade não garante, isoladamente, que todos os documentos encontrados sejam evidências adequadas para responder à pergunta. Essa adequação deverá ser considerada na avaliação dos resultados e na geração das respostas.

In [0]:

# Objetivo:
# Verificar a instalação e a importação do ChromaDB.

# Justificativa:
# Antes de criar o banco vetorial, precisamos confirmar
# que a biblioteca está disponível e funcionando
# corretamente no ambiente Python do notebook.

# Ação:
from importlib.metadata import version

import chromadb

# Consultar a versão instalada da biblioteca.
versao_chromadb = version("chromadb")

print(f"ChromaDB: {versao_chromadb}")
print("Importação realizada com sucesso!")

In [0]:

# Objetivo:
# Recuperar os documentos e embeddings preparados anteriormente.

# Justificativa:
# Após o reinício do kernel, precisamos carregar os arquivos
# persistidos para construir o banco vetorial sem gerar
# novamente os 42.687 embeddings.

# Ação:
from pathlib import Path

import numpy as np
import pandas as pd

# Identificar a pasta principal do projeto no Volume da Fase 4.
DIRETORIO_ATUAL = Path.cwd()
BASE_DIR = Path(
    "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase4"
)

# Definir os caminhos dos arquivos persistidos.
ARQUIVO_DOCUMENTOS = (
    BASE_DIR / "data" / "processed" / "olist_reviews_documentos_rag.jsonl"
)

ARQUIVO_EMBEDDINGS = (
    BASE_DIR / "data" / "processed" / "olist_reviews_embeddings.npy"
)

# Recuperar os documentos com os tipos de dados esperados.
df_documentos_rag = pd.read_json(
    ARQUIVO_DOCUMENTOS,
    orient="records",
    lines=True,
    dtype={
        "documento_id": "str",
        "review_id": "str",
        "order_id": "str",
        "review_score": "int64",
        "review_creation_date": "str",
        "texto_documento": "str"
    },
    convert_dates=False
)

# Recuperar a matriz de embeddings já gerada.
embeddings_documentos_rag = np.load(
    ARQUIVO_EMBEDDINGS,
    allow_pickle=False
)

# Validar a quantidade de documentos e as dimensões dos vetores.
quantidades_correspondem = (
    len(df_documentos_rag) == len(embeddings_documentos_rag)
)

dimensoes_corretas = (
    embeddings_documentos_rag.shape[1] == 384
)

print(f"Documentos carregados: {len(df_documentos_rag):,}")
print(f"Formato dos embeddings: {embeddings_documentos_rag.shape}")
print(f"Quantidades correspondem: {quantidades_correspondem}")
print(f"Dimensões corretas: {dimensoes_corretas}")


### Armazenamento do ChromaDB e reprodutibilidade

O ChromaDB é um banco **persistente e local**. Para evitar problemas de sincronização de arquivos de índice, ele é criado **fora da pasta do projeto**, enquanto os notebooks e scripts permanecem no repositório.

- **Windows:** sem configuração adicional, usa `LOCALAPPDATA/OlistRAG/vector_store`.
- **Outros sistemas ou local personalizado:** defina a variável de ambiente `OLIST_CHROMA_DIR` **antes de executar o notebook**.
- **Outro computador:** prepare o JSONL com o Notebook 02, gere os embeddings (ou disponibilize o `.npy` correspondente) e execute a construção da coleção nas células seguintes. O banco local não precisa ser copiado do computador original.

**Atenção:** a ordem dos documentos no JSONL deve corresponder à ordem dos vetores no arquivo `.npy`. Não use uma pasta sincronizada por OneDrive para o banco. A criação usa `upsert`, de modo que reexecutar com os mesmos IDs não duplica documentos.


**No Databricks:** a célula seguinte define automaticamente `OLIST_CHROMA_DIR` em disco temporário
do compute. O JSONL e o NPY ficam no Volume/S3. Se o compute perder seu cache, recarregue esses artefatos
e execute a construção da coleção a partir desta etapa. Não armazenamos SQLite/HNSW ativo no Volume.


In [0]:

# Objetivo:
# Configurar o banco vetorial local com ChromaDB.

# Justificativa:
# Precisamos de uma coleção persistente para armazenar
# os embeddings das avaliações, seus identificadores,
# textos e metadados, permitindo buscas semânticas futuras.

# Ação:

# Definir o armazenamento local fora da pasta do projeto.
# OLIST_CHROMA_DIR permite configurar o caminho em outro computador.
import os
import tempfile
import hashlib

# Databricks: isolar o banco temporário pela versão exata do JSONL e do NPY.
# Os artefatos de reconstrução permanecem persistidos no Volume/S3.
identidade_base = hashlib.sha256()
for arquivo_base in (ARQUIVO_DOCUMENTOS, ARQUIVO_EMBEDDINGS):
    with arquivo_base.open("rb") as arquivo:
        for bloco in iter(lambda: arquivo.read(1024 * 1024), b""):
            identidade_base.update(bloco)
os.environ["OLIST_CHROMA_DIR"] = str(
    Path(tempfile.gettempdir()) / "olist_fase4" / identidade_base.hexdigest() / "vector_store"
)

caminho_configurado = os.environ.get("OLIST_CHROMA_DIR")
diretorio_local_app = os.environ.get("LOCALAPPDATA")

if caminho_configurado:
    DIRETORIO_BANCO_VETORIAL = Path(caminho_configurado).expanduser().resolve()
elif diretorio_local_app:
    DIRETORIO_BANCO_VETORIAL = (
        Path(diretorio_local_app) / "OlistRAG" / "vector_store"
    ).resolve()
else:
    raise EnvironmentError(
        "Defina OLIST_CHROMA_DIR com uma pasta local para o ChromaDB."
    )

if (
    DIRETORIO_BANCO_VETORIAL == BASE_DIR.resolve()
    or BASE_DIR.resolve() in DIRETORIO_BANCO_VETORIAL.parents
):
    raise ValueError("O ChromaDB deve ficar fora da pasta do projeto.")

# Criar a pasta, caso ainda não exista.
DIRETORIO_BANCO_VETORIAL.mkdir(parents=True, exist_ok=True)

# Inicializar o cliente persistente do ChromaDB.
cliente_chroma = chromadb.PersistentClient(
    path=str(DIRETORIO_BANCO_VETORIAL)
)

# Criar ou recuperar a coleção de avaliações.
colecao_reviews = cliente_chroma.get_or_create_collection(
    name="olist_reviews_rag",
    metadata={"hnsw:space": "cosine"}
)

# Conferir a configuração inicial.
print(f"Diretório do banco: {DIRETORIO_BANCO_VETORIAL}")
print(f"Nome da coleção: {colecao_reviews.name}")
print(f"Documentos armazenados: {colecao_reviews.count()}")

In [0]:

# Objetivo:
# Preparar os dados para inserção na coleção vetorial.

# Justificativa:
# O ChromaDB precisa receber identificadores únicos,
# textos, metadados e embeddings correspondentes.
# Vamos preparar essas estruturas sem alterar a ordem
# original dos documentos.

# Ação:

# Extrair os identificadores e textos dos documentos.
ids_documentos = df_documentos_rag["documento_id"].tolist()
textos_documentos = df_documentos_rag["texto_documento"].tolist()

# Preparar os metadados de cada avaliação.
metadados_documentos = df_documentos_rag[
    ["review_id", "order_id", "review_score", "review_creation_date"]
].to_dict(orient="records")

# Verificar a quantidade de elementos em cada estrutura.
quantidades = {
    "IDs": len(ids_documentos),
    "Textos": len(textos_documentos),
    "Metadados": len(metadados_documentos),
    "Embeddings": len(embeddings_documentos_rag)
}

# Validar a correspondência das quantidades.
quantidades_validas = len(set(quantidades.values())) == 1

# Validar a unicidade dos identificadores.
ids_unicos = len(ids_documentos) == len(set(ids_documentos))

# Conferir se os metadados estão completos.
metadados_completos = all(
    all(valor is not None for valor in metadados.values())
    for metadados in metadados_documentos
)

print("Quantidades preparadas:")
for nome, quantidade in quantidades.items():
    print(f"{nome}: {quantidade:,}")

print(f"\nQuantidades correspondem: {quantidades_validas}")
print(f"Identificadores únicos: {ids_unicos}")
print(f"Metadados completos: {metadados_completos}")

print("\nExemplo do primeiro registro:")
print(f"ID: {ids_documentos[0]}")
print(f"Texto: {textos_documentos[0]}")
print(f"Metadados: {metadados_documentos[0]}")


In [0]:

# Objetivo:
# Inserir os documentos e embeddings da Olist no ChromaDB.

# Justificativa:
# O ChromaDB precisa receber identificadores, textos,
# metadados e embeddings correspondentes.
#
# Vamos explicitar os tipos dos metadados para melhorar
# a compatibilidade com o Pylance.
#
# A inserção será realizada em lotes, utilizando upsert
# para permitir reexecuções sem duplicar identificadores.

# Ação:

from typing import cast
from tqdm.auto import tqdm
from chromadb.api.types import Metadata

# Preparar os metadados com valores de tipos compatíveis.
metadados_documentos = cast(
    list[Metadata],
    [
        {
            "review_id": str(registro.review_id),
            "order_id": str(registro.order_id),
            "review_score": int(str(registro.review_score)),
            "review_creation_date": str(registro.review_creation_date)
        }
        for registro in df_documentos_rag.itertuples(index=False)
    ]
)

# Definir o tamanho dos lotes de inserção.
TAMANHO_LOTE_INSERCAO = 64

# Percorrer os documentos em lotes.
for inicio in tqdm(
    range(0, len(ids_documentos), TAMANHO_LOTE_INSERCAO),
    desc="Inserindo documentos no ChromaDB"
):
    fim = inicio + TAMANHO_LOTE_INSERCAO

    # Inserir os registros e seus embeddings correspondentes.
    colecao_reviews.upsert(
        ids=ids_documentos[inicio:fim],
        documents=textos_documentos[inicio:fim],
        metadatas=metadados_documentos[inicio:fim],
        embeddings=embeddings_documentos_rag[inicio:fim].tolist()
    )

# Conferir a quantidade de documentos armazenados.
quantidade_armazenada = colecao_reviews.count()
quantidade_esperada = len(ids_documentos)

print(f"\nDocumentos no ChromaDB: {quantidade_armazenada:,}")
print(f"Documentos esperados: {quantidade_esperada:,}")
print(f"Quantidade correta: {quantidade_armazenada == quantidade_esperada}")


In [0]:

# Objetivo:
# Validar os registros armazenados no ChromaDB.

# Justificativa:
# A quantidade de documentos armazenados está correta,
# mas precisamos conferir se os identificadores, textos
# e metadados foram preservados durante a inserção.

# Ação:

# Selecionar três posições da base original.
posicoes_amostra = [
    0,
    len(ids_documentos) // 2,
    len(ids_documentos) - 1
]

# Recuperar os identificadores correspondentes.
ids_amostra = [
    ids_documentos[posicao]
    for posicao in posicoes_amostra
]

# Consultar os registros diretamente no ChromaDB.
registros_recuperados = colecao_reviews.get(
    ids=ids_amostra,
    include=["documents", "metadatas", "embeddings"]
)

assert registros_recuperados["documents"] is not None
assert registros_recuperados["metadatas"] is not None
assert registros_recuperados["embeddings"] is not None

# Organizar os registros recuperados por identificador.
indices_recuperados = {
    documento_id: indice
    for indice, documento_id in enumerate(registros_recuperados["ids"])
}

# Comparar os dados originais com os dados armazenados.
for posicao in posicoes_amostra:
    documento_id = ids_documentos[posicao]
    indice = indices_recuperados[documento_id]

    texto_original = textos_documentos[posicao]
    texto_recuperado = registros_recuperados["documents"][indice]

    metadados_originais = metadados_documentos[posicao]
    metadados_recuperados = registros_recuperados["metadatas"][indice]

    embedding_original = embeddings_documentos_rag[posicao]
    embedding_recuperado = registros_recuperados["embeddings"][indice]

    print(f"\nDocumento da posição {posicao:,}")
    print(f"ID: {documento_id}")
    print(f"Texto: {texto_recuperado}")

    print(f"Texto corresponde: {texto_original == texto_recuperado}")
    print(f"Metadados correspondem: {metadados_originais == metadados_recuperados}")
    print(
        "Embedding corresponde:",
        np.allclose(embedding_original, embedding_recuperado)
    )


In [0]:

# Objetivo:
# Realizar a primeira busca semântica nas avaliações da Olist.

# Justificativa:
# Queremos recuperar avaliações relacionadas ao significado
# de uma pergunta, utilizando os embeddings armazenados
# no ChromaDB e o mesmo modelo utilizado na indexação.

# Ação:

from sentence_transformers import SentenceTransformer

# Carregar o mesmo modelo utilizado na geração dos embeddings.
NOME_MODELO = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

modelo_embeddings = SentenceTransformer(NOME_MODELO)

# Definir uma pergunta em linguagem natural.
pergunta = "Quais clientes reclamaram de produtos com defeito?"

# Transformar a pergunta em um embedding.
embedding_pergunta = modelo_embeddings.encode(
    pergunta,
    convert_to_numpy=True
)

# Buscar as cinco avaliações semanticamente mais próximas.
resultados_busca = colecao_reviews.query(
    query_embeddings=[embedding_pergunta.tolist()],
    n_results=5,
    include=["documents", "metadatas", "distances"]
)

# Validar a presença das informações solicitadas.
assert resultados_busca["documents"] is not None
assert resultados_busca["metadatas"] is not None
assert resultados_busca["distances"] is not None

# Exibir os resultados da primeira pergunta.
print(f"Pergunta: {pergunta}")
print(f"Resultados encontrados: {len(resultados_busca['ids'][0])}")

for indice, documento_id in enumerate(resultados_busca["ids"][0]):
    texto = resultados_busca["documents"][0][indice]
    metadados = resultados_busca["metadatas"][0][indice]
    distancia = resultados_busca["distances"][0][indice]

    print(f"\n--- Resultado {indice + 1} ---")
    print(f"ID: {documento_id}")
    print(f"Distância de cosseno: {distancia:.4f}")
    print(f"Nota: {metadados['review_score']}")
    print(f"Texto: {texto}")


In [0]:

# Objetivo:
# Comparar os resultados da busca semântica para
# diferentes temas presentes nas avaliações da Olist.

# Justificativa:
# Uma única pergunta não permite avaliar adequadamente
# o comportamento do mecanismo de recuperação.
# Vamos investigar a relevância dos documentos encontrados
# para perguntas sobre entrega, qualidade e satisfação.

# Ação:

# Definir as perguntas que serão utilizadas nos testes.
perguntas_teste = {
    "Atraso na entrega": (
        "Quais clientes reclamaram de atraso na entrega?"
    ),
    "Produtos com defeito": (
        "Quais clientes receberam produtos quebrados ou com defeito?"
    ),
    "Entrega rápida": (
        "Quais clientes elogiaram a rapidez da entrega?"
    )
}

# Definir a quantidade de resultados por pergunta.
QUANTIDADE_RESULTADOS = 3

# Executar uma busca semântica para cada pergunta.
for tema, pergunta in perguntas_teste.items():

    # Gerar o embedding da pergunta.
    embedding_pergunta = modelo_embeddings.encode(
        pergunta,
        convert_to_numpy=True
    )

    # Consultar os documentos mais próximos no ChromaDB.
    resultados = colecao_reviews.query(
        query_embeddings=[embedding_pergunta.tolist()],
        n_results=QUANTIDADE_RESULTADOS,
        include=["documents", "metadatas", "distances"]
    )

    # Validar a presença dos dados recuperados.
    assert resultados["documents"] is not None
    assert resultados["metadatas"] is not None
    assert resultados["distances"] is not None

    # Exibir os resultados da pergunta.
    print(f"\n{'=' * 60}")
    print(f"TEMA: {tema}")
    print(f"PERGUNTA: {pergunta}")
    print(f"{'=' * 60}")

    for indice, documento_id in enumerate(resultados["ids"][0]):

        texto = resultados["documents"][0][indice]
        metadados = resultados["metadatas"][0][indice]
        distancia = resultados["distances"][0][indice]

        print(f"\nResultado {indice + 1}")
        print(f"ID: {documento_id}")
        print(f"Distância: {distancia:.4f}")
        print(f"Nota: {metadados['review_score']}")
        print(f"Texto: {texto}")


In [0]:

# Objetivo:
# Criar uma função reutilizável para buscar avaliações
# semanticamente relacionadas a uma pergunta.

# Justificativa:
# A função centraliza a geração do embedding da pergunta
# e a consulta ao ChromaDB, facilitando sua reutilização
# no futuro pipeline RAG.

# Ação:

def buscar_avaliacoes(pergunta: str, quantidade: int = 5) -> list[dict]:

    # Validar a pergunta recebida.
    if not pergunta.strip():
        raise ValueError("A pergunta não pode estar vazia.")

    # Validar a quantidade de resultados solicitados.
    if quantidade < 1:
        raise ValueError("A quantidade deve ser maior que zero.")

    # Gerar o embedding da pergunta.
    embedding_pergunta = modelo_embeddings.encode(
        pergunta,
        convert_to_numpy=True
    )

    # Recuperar os documentos semanticamente mais próximos.
    resultados = colecao_reviews.query(
        query_embeddings=[embedding_pergunta.tolist()],
        n_results=quantidade,
        include=["documents", "metadatas", "distances"]
    )

    # Validar os campos retornados pelo ChromaDB.
    assert resultados["documents"] is not None
    assert resultados["metadatas"] is not None
    assert resultados["distances"] is not None

    # Organizar os resultados em uma lista de dicionários.
    avaliacoes_recuperadas = []

    for indice, documento_id in enumerate(resultados["ids"][0]):

        avaliacao = {
            "documento_id": documento_id,
            "texto": resultados["documents"][0][indice],
            "metadados": resultados["metadatas"][0][indice],
            "distancia": resultados["distances"][0][indice]
        }

        avaliacoes_recuperadas.append(avaliacao)

    return avaliacoes_recuperadas


# Confirmar que a função foi definida.
print("Função buscar_avaliacoes() criada com sucesso!")


In [0]:

# Objetivo:
# Testar a função de busca semântica criada na Célula 26.

# Justificativa:
# Precisamos confirmar que a função executa a consulta
# ao ChromaDB e retorna os documentos no formato esperado,
# antes de reutilizá-la no futuro pipeline RAG.

# Ação:

# Definir uma pergunta para o teste.
pergunta_teste = "Quais consumidores receberam produtos com defeito?"

# Executar a busca utilizando nossa função.
avaliacoes_encontradas = buscar_avaliacoes(
    pergunta=pergunta_teste,
    quantidade=3
)

# Definir os campos esperados em cada resultado.
campos_esperados = {
    "documento_id",
    "texto",
    "metadados",
    "distancia"
}

# Validar a quantidade e a estrutura dos resultados.
quantidade_correta = len(avaliacoes_encontradas) == 3

estrutura_correta = all(
    campos_esperados.issubset(avaliacao.keys())
    for avaliacao in avaliacoes_encontradas
)

# Exibir as verificações.
print(f"Pergunta: {pergunta_teste}")
print(f"Resultados encontrados: {len(avaliacoes_encontradas)}")
print(f"Quantidade correta: {quantidade_correta}")
print(f"Estrutura correta: {estrutura_correta}")

# Exibir os documentos recuperados.
for indice, avaliacao in enumerate(avaliacoes_encontradas, start=1):

    print(f"\n--- Avaliação {indice} ---")
    print(f"ID: {avaliacao['documento_id']}")
    print(f"Distância: {avaliacao['distancia']:.4f}")
    print(f"Nota: {avaliacao['metadados']['review_score']}")
    print(f"Texto: {avaliacao['texto']}")


In [0]:

# Objetivo:
# Preparar a estrutura do projeto para armazenar
# funções Python reutilizáveis.

# Justificativa:
# O módulo de busca vetorial será utilizado por outros
# componentes do projeto, especialmente pelo Notebook 04.

# Ação:

# Definir o diretório destinado aos módulos Python.
DIRETORIO_CODIGO = BASE_DIR / "src"

# Criar a pasta caso ela ainda não exista.
DIRETORIO_CODIGO.mkdir(
    parents=True,
    exist_ok=True
)

# Confirmar a criação do diretório.
print(f"Diretório: {DIRETORIO_CODIGO}")
print(f"Diretório existente: {DIRETORIO_CODIGO.is_dir()}")


In [0]:

# Objetivo:
# Criar um módulo Python reutilizável para a busca vetorial.

# Justificativa:
# O módulo permitirá utilizar a recuperação semântica
# em outros notebooks, sem depender das variáveis
# criadas durante a exploração do Notebook 03.

# Ação:

codigo_modulo = '''\
import os
from pathlib import Path

import chromadb
from sentence_transformers import SentenceTransformer


# Configurações do projeto.
BASE_DIR = Path(__file__).resolve().parent.parent

# Usar o mesmo armazenamento configurável do notebook.
caminho_configurado = os.environ.get("OLIST_CHROMA_DIR")
diretorio_local_app = os.environ.get("LOCALAPPDATA")

if caminho_configurado:
    DIRETORIO_BANCO_VETORIAL = Path(caminho_configurado).expanduser().resolve()
elif diretorio_local_app:
    DIRETORIO_BANCO_VETORIAL = (
        Path(diretorio_local_app) / "OlistRAG" / "vector_store"
    ).resolve()
else:
    raise EnvironmentError(
        "Defina OLIST_CHROMA_DIR para localizar o banco vetorial."
    )

NOME_MODELO = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

NOME_COLECAO = "olist_reviews_rag"


def inicializar_busca_vetorial():

    # Carregar o mesmo modelo utilizado na geração dos embeddings.
    modelo = SentenceTransformer(NOME_MODELO)

    # Conectar ao banco vetorial persistente.
    cliente = chromadb.PersistentClient(
        path=str(DIRETORIO_BANCO_VETORIAL)
    )

    # Acessar a coleção existente, sem recriá-la.
    colecao = cliente.get_collection(
        name=NOME_COLECAO
    )

    return modelo, colecao


def buscar_avaliacoes(
    pergunta: str,
    modelo,
    colecao,
    quantidade: int = 5
) -> list[dict]:

    # Validar os parâmetros recebidos.
    if not pergunta.strip():
        raise ValueError("A pergunta não pode estar vazia.")

    if quantidade < 1:
        raise ValueError("A quantidade deve ser maior que zero.")

    # Gerar o embedding da pergunta.
    embedding_pergunta = modelo.encode(
        pergunta,
        convert_to_numpy=True
    )

    # Consultar o banco vetorial.
    resultados = colecao.query(
        query_embeddings=[embedding_pergunta.tolist()],
        n_results=quantidade,
        include=["documents", "metadatas", "distances"]
    )

    # Validar os campos retornados.
    assert resultados["documents"] is not None
    assert resultados["metadatas"] is not None
    assert resultados["distances"] is not None

    # Organizar os documentos recuperados.
    avaliacoes_recuperadas = []

    for indice, documento_id in enumerate(resultados["ids"][0]):

        avaliacao = {
            "documento_id": documento_id,
            "texto": resultados["documents"][0][indice],
            "metadados": resultados["metadatas"][0][indice],
            "distancia": resultados["distances"][0][indice]
        }

        avaliacoes_recuperadas.append(avaliacao)

    return avaliacoes_recuperadas
'''

# Definir o caminho do arquivo Python.
# Nome específico da adaptação para preservar o busca_vetorial.py já existente.
ARQUIVO_MODULO = DIRETORIO_CODIGO / "busca_vetorial_databricks.py"

# Preservar versões existentes com alterações próprias.
if ARQUIVO_MODULO.exists():
    codigo_existente = ARQUIVO_MODULO.read_text(encoding="utf-8")
    if codigo_existente != codigo_modulo:
        # O módulo já pode ter sido criado e validado em execução anterior.
        # Nesse caso, não sobrescrever sem inspeção.
        if (
            'os.environ.get("OLIST_CHROMA_DIR")' not in codigo_existente
            or "def inicializar_busca_vetorial(" not in codigo_existente
            or "def buscar_avaliacoes(" not in codigo_existente
        ):
            raise ValueError(
                "O módulo existente difere da versão esperada. "
                "Revise src/busca_vetorial.py antes de continuar."
            )
        print("Módulo existente com caminho configurável preservado.")
else:
    ARQUIVO_MODULO.write_text(codigo_modulo, encoding="utf-8")

# Confirmar a criação do módulo.
print(f"Arquivo criado: {ARQUIVO_MODULO}")
print(f"Arquivo existente: {ARQUIVO_MODULO.is_file()}")


In [0]:
# Objetivo:
# Validar o módulo reutilizável com o banco vetorial configurado.

# Justificativa:
# O Notebook 04 usará este módulo, e não as variáveis internas
# deste notebook. Precisamos testar a integração real.

# Ação:
import importlib.util
import sys
import shutil
import tempfile

# Python importa uma cópia local; o arquivo persistido no Volume é preservado.
caminho_modulo_local = Path(tempfile.mkdtemp(prefix="olist_modulo_busca_")) / ARQUIVO_MODULO.name
shutil.copyfile(ARQUIVO_MODULO, caminho_modulo_local)
especificacao = importlib.util.spec_from_file_location(
    "olist_busca_vetorial_databricks", caminho_modulo_local
)
if especificacao is None or especificacao.loader is None:
    raise ImportError(f"Não foi possível carregar o módulo: {ARQUIVO_MODULO}")
modulo_busca = importlib.util.module_from_spec(especificacao)
sys.modules[especificacao.name] = modulo_busca
especificacao.loader.exec_module(modulo_busca)

# Confirmar que o módulo aponta para a coleção preparada acima.
diretorio_modulo = modulo_busca.DIRETORIO_BANCO_VETORIAL.resolve()
assert diretorio_modulo == DIRETORIO_BANCO_VETORIAL.resolve(), (
    "O módulo e o notebook apontam para bancos diferentes."
)

modelo_modulo, colecao_modulo = modulo_busca.inicializar_busca_vetorial()
quantidade_documentos = colecao_modulo.count()
assert quantidade_documentos == len(df_documentos_rag)
assert modelo_modulo.get_sentence_embedding_dimension() == 384

pergunta_teste = "Quais clientes reclamaram de produtos com defeito?"
avaliacoes_teste = modulo_busca.buscar_avaliacoes(
    pergunta=pergunta_teste,
    modelo=modelo_modulo,
    colecao=colecao_modulo,
    quantidade=5
)

campos_esperados = {"documento_id", "texto", "metadados", "distancia"}
assert len(avaliacoes_teste) == 5
assert all(campos_esperados.issubset(item) for item in avaliacoes_teste)

print("=== VALIDAÇÃO DO MÓDULO ===")
print("Diretório:", diretorio_modulo)
print("Documentos:", quantidade_documentos)
print("Avaliações recuperadas:", len(avaliacoes_teste))
for indice, avaliacao in enumerate(avaliacoes_teste, start=1):
    print(f"\n--- Avaliação {indice} ---")
    print("Documento ID:", avaliacao["documento_id"])
    print("Nota:", avaliacao["metadados"]["review_score"])
    print("Distância:", round(avaliacao["distancia"], 4))
    print("Texto:", avaliacao["texto"])
print("\nVALIDAÇÃO DO MÓDULO: APROVADA")


## Conclusão e próximos passos

### Resultados alcançados

Neste notebook, concluímos a implementação e a validação da etapa de **geração de embeddings e busca vetorial**, que constitui a base de recuperação de informações do projeto *Voice of Customer Intelligence com RAG*.

As principais entregas foram:

- **42.687 documentos de avaliações de clientes** carregados e processados.
- Geração de embeddings com o modelo `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`, produzindo vetores de **384 dimensões**.
- Persistência dos embeddings em arquivo `.npy`, permitindo sua reutilização.
- Criação e validação de uma coleção no **ChromaDB**, utilizando similaridade por cosseno.
- Configuração do armazenamento vetorial em um diretório local externo ao OneDrive, com possibilidade de personalização pela variável de ambiente `OLIST_CHROMA_DIR`.
- Implementação do módulo reutilizável `src/busca_vetorial.py`, responsável por inicializar o modelo, acessar a coleção e recuperar avaliações semanticamente semelhantes a uma pergunta.
- **Execução integral do notebook e validação funcional do módulo**, confirmando a recuperação de documentos e seus respectivos metadados.

### Limitações da recuperação semântica

A busca vetorial retorna **documentos candidatos potencialmente relevantes**, ordenados pela proximidade entre os embeddings da pergunta e das avaliações.

Entretanto, **similaridade semântica não garante relevância factual**. Algumas avaliações recuperadas podem abordar assuntos relacionados, mas não responder diretamente à pergunta apresentada.

Por isso, os documentos recuperados devem ser tratados como **evidências a serem analisadas**, e não como respostas automaticamente verificadas.

### Continuidade do projeto

Com a infraestrutura de recuperação validada, a próxima etapa será a **integração com um modelo de linguagem (LLM)**, a ser desenvolvida no Notebook 04.

Essa integração deverá:

1. Receber perguntas formuladas em linguagem natural.
2. Recuperar avaliações relevantes por meio de `src/busca_vetorial.py`.
3. Incorporar os documentos recuperados ao contexto enviado ao LLM.
4. Gerar respostas fundamentadas nas avaliações disponíveis.
5. Identificar limitações das evidências e evitar afirmações não sustentadas pelos documentos.

Posteriormente, será necessário avaliar a qualidade das respostas, a pertinência dos documentos recuperados e o comportamento do sistema diante de perguntas sem evidências suficientes.

### Reprodutibilidade

O Notebook 03 foi executado integralmente e validado no ambiente de desenvolvimento utilizado neste projeto.

Para sua execução em outros computadores, será necessário disponibilizar os dados de entrada, instalar as dependências e reconstruir a coleção vetorial localmente.

As instruções de instalação, configuração e execução deverão ser consolidadas no arquivo `README.md` do repositório.

---

**Status da etapa:** concluída e validada funcionalmente.

**Próxima etapa:** Notebook 04 — Integração do LLM com a arquitetura RAG.

### Observações para esta versão Databricks
As conclusões e o status acima foram mantidos do notebook original; não comprovam execução desta adaptação.
Nesta versão, confirme o êxito de todas as células e a mensagem de validação do módulo no seu workspace.
A etapa original de geração recalcula os embeddings e grava o NPY no Volume, como já fazia no arquivo original.
O módulo desta adaptação é salvo em `src/busca_vetorial_databricks.py`, preservando o arquivo
`src/busca_vetorial.py` que já existe no projeto. O Notebook 04 deverá usar o novo nome e as assinaturas
validadas aqui. O banco ativo é temporário e pode ser reconstruído pelos arquivos persistidos.
